In [ ]:
"""
Sliding-Window-Methodik – Visualisierung als MP4
=================================================
Inhalt:
  Frame 1 : Patientenübersicht (Demographie + Vorerkrankungen)
  Frame 2–4: 3 Hops des 2h-Kontextfensters über den ICU-Aufenthalt
             mit MAP, Medikamentengabe, Modellvorhersage, Ground Truth
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.gridspec import GridSpec
import tempfile, subprocess
from pathlib import Path
from PIL import Image as PILImage

# ─── Fonts & Style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

# ─── Farbpalette ──────────────────────────────────────────────────────────────
BG        = "#8f959e"
PANEL_BG  = "#161b22"
BORDER    = "#30363d"
TEXT_MAIN = "#e6edf3"
TEXT_MUTED= "#8b949e"
ACCENT    = "#58a6ff"
GREEN     = "#3fb950"
RED       = "#ff7b72"
ORANGE    = "#f0883e"

CAT_COLORS = {
    "Kristalloide": "#4fc3f7",
    "Elektrolyte":  "#ce93d8",
    "Antibiotika":  "#a5d6a7",
}

# ─── Fake-Daten ───────────────────────────────────────────────────────────────
np.random.seed(42)
N_STUNDEN = 24
ZEIT = np.linspace(0, N_STUNDEN, 200)

def make_map(t):
    base = 75 + 8 * np.sin(t * 0.3) - 3 * np.sin(t * 1.1)
    dip1 = -25 * np.exp(-0.5 * ((t - 9)  / 0.8) ** 2)
    dip2 = -18 * np.exp(-0.5 * ((t - 19) / 1.0) ** 2)
    noise = np.random.normal(0, 2, len(t))
    return np.clip(base + dip1 + dip2 + noise, 35, 130)

MAP_WERTE = make_map(ZEIT)

MED_ZEIT = np.arange(0, N_STUNDEN + 0.5, 0.5)   # alle 30 min
rng = np.random.default_rng(7)
MEDS = {
    "Kristalloide": (rng.random(len(MED_ZEIT)) < 0.35).astype(int),
    "Elektrolyte":  (rng.random(len(MED_ZEIT)) < 0.20).astype(int),
    "Antibiotika":  (rng.random(len(MED_ZEIT)) < 0.12).astype(int),
}

GT_START = 19.0
GT_END   = 21.0

VORHERSAGEN = [
    {"kontext_ende": 17.0, "ziel_start": 19.0, "ziel_ende": 21.0},
    {"kontext_ende": 17.5, "ziel_start": 19.5, "ziel_ende": 21.5},
    {"kontext_ende": 18.0, "ziel_start": 19.0, "ziel_ende": 21.0},
]

PATIENT = {
    "id":          "ICU-202300",
    "geschlecht":  "Männlich",
    "alter":       67,
    "ethnie":      "Kaukasisch",
    "groesse_cm":  178,
    "gewicht_kg":  84,
    # Vorerkrankungen relevant für Hypotension
    "vorerkrankungen": {
        "Arterielle Hypertonie": True,
        "Diabetes mellitus":     True,
        "Herzinsuffizienz":      True,
        "Chronische Niereninsuffizienz": False,
        "Koronare Herzkrankheit":True,
        "Sepsis (Voraufnahme)":  False,
    },
}
BMI = PATIENT["gewicht_kg"] / (PATIENT["groesse_cm"] / 100) ** 2

FENSTER_GROESSE = 2.0     # Stunden
HOP_ZENTREN     = [6.0, 12.0, 18.0]

# ─── Hilfsfunktionen ──────────────────────────────────────────────────────────

def dunkles_bg(fig, axes):
    fig.patch.set_facecolor(BG)
    for ax in axes:
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors=TEXT_MUTED, labelsize=9)
        ax.xaxis.label.set_color(TEXT_MUTED)
        ax.yaxis.label.set_color(TEXT_MUTED)
        for spine in ax.spines.values():
            spine.set_edgecolor(BORDER)
        ax.title.set_color(TEXT_MAIN)


def fenster_markieren(ax, mitte, groesse, ymax, beschriftung=True):
    lo, hi = mitte - groesse / 2, mitte + groesse / 2
    ax.axvspan(lo, hi, color=ACCENT, alpha=0.13, zorder=0)
    ax.axvline(lo, color=ACCENT, lw=1.2, ls="--", alpha=0.7)
    ax.axvline(hi, color=ACCENT, lw=1.2, ls="--", alpha=0.7)
    if beschriftung:
        ax.annotate("", xy=(hi, ymax * 0.92), xytext=(lo, ymax * 0.92),
                    arrowprops=dict(arrowstyle="<->", color=ACCENT, lw=1.4))
        ax.text((lo + hi) / 2, ymax * 0.96, "2h Kontextfenster",
                ha="center", fontsize=8, color=ACCENT, fontweight="bold")


def vorhersage_pfeil(ax, kontext_ende, ziel_start, ziel_ende, y):
    ax.annotate("", xy=(ziel_start, y), xytext=(kontext_ende, y),
                arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.8,
                                connectionstyle="arc3,rad=-0.28"),
                zorder=8)
    ax.axvspan(ziel_start, ziel_ende, color=ORANGE, alpha=0.18, zorder=1)


# ═══════════════════════════════════════════════════════════════════════════════
#  Frame 1 – Patientenübersicht
# ═══════════════════════════════════════════════════════════════════════════════

def frame_metadaten(speicherpfad):
    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    gs  = GridSpec(1, 2, figure=fig,
                   left=0.04, right=0.96, top=0.90, bottom=0.08, wspace=0.06)

    ax_l = fig.add_subplot(gs[0, 0])
    ax_r = fig.add_subplot(gs[0, 1])
    for ax in (ax_l, ax_r):
        ax.set_facecolor(PANEL_BG)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
        for sp in ax.spines.values(): sp.set_edgecolor(BORDER)

    fig.text(0.5, 0.96,
             "Gleitendes Kontextfenster — Patientenübersicht",
             ha="center", va="top", fontsize=18,
             fontweight="bold", color=TEXT_MAIN)
    fig.text(0.5, 0.91,
             f"ICU-Aufenthalt: {PATIENT['id']}",
             ha="center", va="top", fontsize=12, color=TEXT_MUTED)

    # ── Links: Demographie ──
    ax_l.text(0.5, 0.97, "Demographie", ha="center", va="top",
              fontsize=14, fontweight="bold", color=ACCENT)

    demo = [
        ("Geschlecht",  PATIENT["geschlecht"]),
        ("Alter",       f"{PATIENT['alter']} Jahre"),
        ("Ethnie",      PATIENT["ethnie"]),
        ("Größe",       f"{PATIENT['groesse_cm']} cm"),
        ("Gewicht",     f"{PATIENT['gewicht_kg']} kg"),
        ("BMI",         f"{BMI:.1f} kg/m²"),
    ]
    y = 0.87
    for label, wert in demo:
        ax_l.text(0.12, y, label, fontsize=12, color=TEXT_MUTED, va="top")
        ax_l.text(0.88, y, wert,  fontsize=12, color=TEXT_MAIN,
                  va="top", ha="right", fontweight="bold")
        ax_l.axhline(y - 0.005, xmin=0.1, xmax=0.9, color=BORDER, lw=0.6)
        y -= 0.12

    # ── Rechts: Vorerkrankungen ──
    ax_r.text(0.5, 0.97, "Relevante Vorerkrankungen\n(Risikofaktoren für Hypotonie)",
              ha="center", va="top", fontsize=13, fontweight="bold", color=ACCENT)

    items   = list(PATIENT["vorerkrankungen"].items())
    box_w, box_h = 0.42, 0.115
    xs = [0.04, 0.54]
    ys = np.linspace(0.82, 0.18, (len(items) + 1) // 2)

    for idx, (name, vorhanden) in enumerate(items):
        r, c = idx // 2, idx % 2
        bx, by = xs[c], ys[r]
        fc = "#2d1f1f" if vorhanden else "#1a2130"
        ec = RED       if vorhanden else BORDER
        ax_r.add_patch(FancyBboxPatch(
            (bx, by - box_h), box_w, box_h,
            boxstyle="round,pad=0.01",
            facecolor=fc, edgecolor=ec, lw=2.0, zorder=2))
        symbol  = "✓" if vorhanden else "–"
        sym_col = RED if vorhanden else TEXT_MUTED
        ax_r.text(bx + 0.05, by - box_h / 2, symbol,
                  fontsize=14, color=sym_col, va="center", fontweight="bold")
        ax_r.text(bx + 0.13, by - box_h / 2, name,
                  fontsize=9,
                  color=TEXT_MAIN  if vorhanden else TEXT_MUTED,
                  va="center",
                  fontweight="bold" if vorhanden else "normal")

    # Legende
    for fy, sym, txt, col in [
        (0.08, "✓", "Vorhanden", RED),
        (0.03, "–", "Nicht vorhanden", TEXT_MUTED),
    ]:
        ax_r.text(0.55, fy, sym, fontsize=11, color=col, va="bottom", fontweight="bold")
        ax_r.text(0.62, fy, txt, fontsize=9,  color=TEXT_MUTED, va="bottom")

    _speichern(fig, speicherpfad)


# ═══════════════════════════════════════════════════════════════════════════════
#  Frames 2–4 – Sliding Window Hops
# ═══════════════════════════════════════════════════════════════════════════════

def frame_hop(hop_idx, mitte, zeige_vorhersage, speicherpfad):
    w_lo = mitte - FENSTER_GROESSE / 2
    w_hi = mitte + FENSTER_GROESSE / 2

    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    gs  = GridSpec(4, 1, figure=fig,
                   left=0.08, right=0.95, top=0.90, bottom=0.08, hspace=0.55)

    ax_map  = fig.add_subplot(gs[0])
    ax_med  = fig.add_subplot(gs[1])
    ax_pred = fig.add_subplot(gs[2])
    ax_gt   = fig.add_subplot(gs[3])

    dunkles_bg(fig, [ax_map, ax_med, ax_pred, ax_gt])

    for ax in (ax_map, ax_med, ax_pred, ax_gt):
        ax.set_xlim(0, N_STUNDEN)
        ax.tick_params(axis='x', colors=TEXT_MUTED, labelsize=9)
        ax.tick_params(axis='y', colors=TEXT_MUTED, labelsize=9)
        for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
        # Vergangener Bereich leicht hervorgehoben
        ax.axvspan(0, w_hi, color="#1c2535", alpha=0.35, zorder=0)

    for ax in (ax_map, ax_med, ax_pred):
        ax.set_xticks([])

    ax_gt.set_xlabel("Stunden seit ICU-Aufnahme", color=TEXT_MUTED, fontsize=10)
    ax_gt.set_xticks(range(0, N_STUNDEN + 1, 2))

    # ── Titel ──
    fig.text(0.5, 0.96,
             f"Gleitendes Kontextfenster — Schritt {hop_idx + 1}/3"
             f"   (Fenster: {w_lo:.0f}h – {w_hi:.0f}h)",
             ha="center", va="top", fontsize=16, fontweight="bold", color=TEXT_MAIN)
    fig.text(0.5, 0.92,
             f"ICU-Aufenthalt: {PATIENT['id']}  |  2h Kontext  →  Modellausgabe",
             ha="center", va="top", fontsize=10, color=TEXT_MUTED)

    # Schritt-Indikatoren (kleine Punkte oben rechts)
    for i, hc in enumerate(HOP_ZENTREN):
        col  = ACCENT    if i == hop_idx else BORDER
        size = 12        if i == hop_idx else 8
        fig.text(0.87 + i * 0.04, 0.945, "●",
                 ha="center", va="center", fontsize=size, color=col)

    # ══ Panel 1: MAP ═════════════════════════════════════════════════════════
    ax_map.plot(ZEIT, MAP_WERTE, color="#607d8b", lw=0.9, alpha=0.45)
    maske = (ZEIT >= w_lo) & (ZEIT <= w_hi)
    ax_map.plot(ZEIT[maske], MAP_WERTE[maske], color=TEXT_MAIN, lw=2.4)
    ax_map.axhline(65, color=RED, lw=1.5, ls="--", alpha=0.7,
                   label="Hypotonie-Schwelle (65 mmHg)")
    fenster_markieren(ax_map, mitte, FENSTER_GROESSE, 125)
    ax_map.set_ylim(35, 125)
    ax_map.set_ylabel("MAP\n(mmHg)", color=TEXT_MUTED, fontsize=9)
    ax_map.set_title("Mittlerer Arterieller Druck (MAP)", loc="left",
                     fontsize=11, color=TEXT_MAIN, fontweight="bold", pad=4)
    ax_map.legend(loc="upper right", fontsize=8,
                  facecolor=PANEL_BG, edgecolor=BORDER, labelcolor=TEXT_MUTED)

    # ══ Panel 2: Medikamentengabe ═════════════════════════════════════════════
    y_pos = {"Kristalloide": 3, "Elektrolyte": 2, "Antibiotika": 1}
    for name, werte in MEDS.items():
        col = CAT_COLORS[name]
        for t_idx, (t, v) in enumerate(zip(MED_ZEIT, werte)):
            if v:
                im_fenster = w_lo <= t <= w_hi
                ax_med.bar(t, 0.6, width=0.44,
                           bottom=y_pos[name] - 0.3,
                           color=col, alpha=0.85 if im_fenster else 0.22,
                           zorder=3)

    fenster_markieren(ax_med, mitte, FENSTER_GROESSE, 4.5, beschriftung=False)
    ax_med.set_ylim(0.2, 4.5)
    ax_med.set_yticks([1, 2, 3])
    ax_med.set_yticklabels(["Antibiotika", "Elektrolyte", "Kristalloide"],
                            fontsize=8, color=TEXT_MUTED)
    ax_med.set_title("Medikamentengabe (0/1 je 30-min-Intervall)",
                     loc="left", fontsize=11, color=TEXT_MAIN,
                     fontweight="bold", pad=4)
    handles = [mpatches.Patch(facecolor=CAT_COLORS[n], label=n, edgecolor="none")
               for n in CAT_COLORS]
    ax_med.legend(handles=handles, loc="upper right", fontsize=8, ncol=3,
                  facecolor=PANEL_BG, edgecolor=BORDER, labelcolor=TEXT_MUTED)

    # ══ Panel 3: Modellvorhersage ═════════════════════════════════════════════
    ax_pred.set_ylim(0, 2)
    ax_pred.set_yticks([])
    ax_pred.set_title("Modellvorhersage: Katecholamin-Initiierung",
                      loc="left", fontsize=11, color=TEXT_MAIN,
                      fontweight="bold", pad=4)

    if zeige_vorhersage:
        for v in VORHERSAGEN:
            if v["kontext_ende"] <= w_hi:
                vorhersage_pfeil(ax_pred, v["kontext_ende"],
                                 v["ziel_start"], v["ziel_ende"], y=1.0)
                ax_pred.plot(v["kontext_ende"], 1.0, "o",
                             color=ORANGE, ms=7,
                             markeredgecolor=BG, markeredgewidth=1.5, zorder=9)
        ax_pred.text(GT_START + 0.15, 1.68,
                     "Vorhergesagtes\nRisikoFenster",
                     fontsize=8, color=ORANGE, va="top")
    else:
        ax_pred.text(N_STUNDEN / 2, 1.0,
                     "— Kein Ereignis im aktuellen Horizont —",
                     ha="center", va="center", fontsize=9,
                     color=TEXT_MUTED, style="italic")

    fenster_markieren(ax_pred, mitte, FENSTER_GROESSE, 2, beschriftung=False)

    # ══ Panel 4: Ground Truth ════════════════════════════════════════════════
    ax_gt.set_ylim(0, 2)
    ax_gt.set_yticks([])
    ax_gt.set_title("Ground Truth: Katecholamin-Initiierungsfenster",
                    loc="left", fontsize=11, color=TEXT_MAIN,
                    fontweight="bold", pad=4)

    if w_hi >= GT_START - 1:
        ax_gt.add_patch(Rectangle(
            (GT_START, 0.3), GT_END - GT_START, 1.4,
            facecolor=RED, edgecolor=RED, alpha=0.40, lw=2, zorder=3))
        ax_gt.text((GT_START + GT_END) / 2, 1.68,
                   "Vasopressor-Gabe",
                   ha="center", fontsize=8, color=RED, fontweight="bold")
    else:
        ax_gt.text(N_STUNDEN / 2, 1.0,
                   "— Noch nicht eingetreten —",
                   ha="center", va="center", fontsize=9,
                   color=TEXT_MUTED, style="italic")

    fenster_markieren(ax_gt, mitte, FENSTER_GROESSE, 2, beschriftung=False)

    # Aktueller Zeitpunkt (grüne Linie)
    for ax in (ax_map, ax_med, ax_pred, ax_gt):
        ax.axvline(w_hi, color=GREEN, lw=2.2, alpha=0.95, zorder=10)
    ax_map.text(w_hi + 0.25, 120,
                f"t = {w_hi:.0f}h",
                fontsize=8, color=GREEN, va="top", fontweight="bold")

    _speichern(fig, speicherpfad)


# ═══════════════════════════════════════════════════════════════════════════════
#  Übergangsframe
# ═══════════════════════════════════════════════════════════════════════════════

def frame_uebergang(von_mitte, zu_mitte, speicherpfad):
    fig, ax = plt.subplots(figsize=(16, 9))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG); ax.axis("off")
    ax.set_xlim(0, 1);    ax.set_ylim(0, 1)

    von_lo = von_mitte - FENSTER_GROESSE / 2
    von_hi = von_mitte + FENSTER_GROESSE / 2
    zu_lo  = zu_mitte  - FENSTER_GROESSE / 2
    zu_hi  = zu_mitte  + FENSTER_GROESSE / 2

    ax.text(0.5, 0.60,
            f"Fenster springt vor:",
            ha="center", va="center", fontsize=16, color=TEXT_MUTED)
    ax.text(0.5, 0.50,
            f"{von_lo:.0f}h – {von_hi:.0f}h   ⟶   {zu_lo:.0f}h – {zu_hi:.0f}h",
            ha="center", va="center", fontsize=22,
            color=ACCENT, fontweight="bold")
    ax.text(0.5, 0.38,
            "Gleitendes Kontextfenster",
            ha="center", va="center", fontsize=13, color=TEXT_MUTED)

    _speichern(fig, speicherpfad)


# ═══════════════════════════════════════════════════════════════════════════════
#  Speichern (gemeinsam, feste Auflösung 1920×1080)
# ═══════════════════════════════════════════════════════════════════════════════

TARGET_W, TARGET_H = 1920, 1080

def _speichern(fig, pfad):
    """Speichert Figure als PNG mit genau TARGET_W × TARGET_H Pixeln."""
    dpi = 120
    fig.set_size_inches(TARGET_W / dpi, TARGET_H / dpi)
    plt.savefig(pfad, dpi=dpi, facecolor=fig.get_facecolor(),
                bbox_inches=None)
    plt.close(fig)


# ═══════════════════════════════════════════════════════════════════════════════
#  Video zusammensetzen – ausschließlich via subprocess ffmpeg
#  (robusteste Methode, keine Codec-Probleme, öffnet überall)
# ═══════════════════════════════════════════════════════════════════════════════

def video_erstellen(frame_pfade, ausgabe_pfad, fps=4):
    """
    Schreibt alle PNG-Frames als H.264 MP4 (yuv420p) via ffmpeg CLI.
    Das erzeugte Video öffnet sich in VLC, Windows Media Player, QuickTime etc.
    """
    # Frames in tmp-Ordner kopieren (brauchen lückenlose Nummerierung)
    tmp = Path(tempfile.mkdtemp())
    for i, src in enumerate(frame_pfade):
        img = PILImage.open(src).convert("RGB").resize(
            (TARGET_W, TARGET_H), PILImage.LANCZOS)
        img.save(tmp / f"frame_{i:05d}.png")

    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-i",         str(tmp / "frame_%05d.png"),
        "-c:v",       "libx264",
        "-preset",    "slow",
        "-crf",       "18",          # hohe Qualität
        "-pix_fmt",   "yuv420p",     # Pflicht für breite Kompatibilität
        "-movflags",  "+faststart",  # Web-Streaming-optimiert
        ausgabe_pfad,
    ]

    ergebnis = subprocess.run(cmd, capture_output=True, text=True)

    # Aufräumen
    for f in tmp.iterdir(): f.unlink()
    tmp.rmdir()

    if ergebnis.returncode != 0:
        raise RuntimeError(
            f"ffmpeg schlug fehl:\n{ergebnis.stderr[-800:]}\n\n"
            "Stelle sicher, dass ffmpeg installiert ist:\n"
            "  conda install -c conda-forge ffmpeg"
        )
    print(f"✅  Video gespeichert: {ausgabe_pfad}")


# ═══════════════════════════════════════════════════════════════════════════════
#  Hauptprogramm
# ═══════════════════════════════════════════════════════════════════════════════

def main(ausgabe_pfad="sliding_window_methodik.mp4", fps=4):
    tmp = Path(tempfile.mkdtemp())
    frames = []

    # Frame 1: Patientenmetadaten (~3,5 s)
    print("Rendere Metadaten-Frame …")
    p = tmp / "f000_meta.png"
    frame_metadaten(p)
    frames += [str(p)] * (fps * 4)   # 4 Sekunden

    # Frames 2–4: Hops
    for hop_i, mitte in enumerate(HOP_ZENTREN):
        print(f"Rendere Hop {hop_i + 1}/3 (Fenster: "
              f"{mitte - FENSTER_GROESSE/2:.0f}h – {mitte + FENSTER_GROESSE/2:.0f}h) …")

        # Übergangsframe
        if hop_i > 0:
            tp = tmp / f"trans_{hop_i:02d}.png"
            frame_uebergang(HOP_ZENTREN[hop_i - 1], mitte, tp)
            frames += [str(tp)] * (fps * 2)   # 2 Sekunden

        # Hop-Frame
        zeige_pred = (mitte >= 18.0)
        hp = tmp / f"hop_{hop_i:02d}.png"
        frame_hop(hop_i, mitte, zeige_pred, hp)
        frames += [str(hp)] * (fps * 5)   # 5 Sekunden

    # Letzten Frame etwas länger halten
    frames += [frames[-1]] * (fps * 2)

    print(f"Erzeuge MP4 aus {len(frames)} Frames …")
    video_erstellen(frames, ausgabe_pfad, fps=fps)

    # Aufräumen
    for f in tmp.iterdir(): f.unlink()
    tmp.rmdir()


if __name__ == "__main__":
    main()